# Comparación de Random Forest y Extra Trees en ASSISTments

Notebook complementario de la práctica 1-2. Compara tres corridas sobre el mismo CSV procesado de 3 000 estudiantes y la misma partición oficial: Random Forest de la práctica, Random Forest con hiperparámetros modificados y Extra Trees, algoritmo incluido en el estudio original [1].

La comparación conserva las 722 variables, las etiquetas score y la división source_split. Los parámetros de la segunda corrida se fijan manualmente; no se ejecuta búsqueda automática. Las métricas de entrenamiento describen ajuste y las de evaluation se calculan sobre los registros reservados.


## 1. Cargar el CSV preparado

Sube `assistments_rf_dataset.csv.gz` desde Colab. Esta notebook compara modelos sobre el mismo archivo preparado; no vuelve a integrar archivos crudos.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from google.colab import files

files.upload()  # Selecciona assistments_rf_dataset.csv.gz
DATASET_CSV = Path('/content/assistments_rf_dataset.csv.gz')
df = pd.read_csv(DATASET_CSV, low_memory=False)

print(f'Filas: {len(df):,} | estudiantes: {df.student_id.nunique():,} | columnas: {df.shape[1]:,}')
display(df['source_split'].value_counts().rename_axis('partición').to_frame('filas'))
display(df['score'].value_counts().sort_index().rename_axis('score').to_frame('filas'))


## 2. Definir X, y y la partición

`score` es la variable dependiente. Se excluye de `X`, junto con `student_id`, `source_split`, `unit_test_assignment_log_id` y `problem_id`. Se usan las particiones oficiales `train` y `evaluation`; no se crea una partición de respaldo.

La revisión de estudiantes compartidos se mantiene porque limita la interpretación de evaluation como prueba de estudiantes completamente nuevos.


In [ ]:
# score es la etiqueta; los IDs y source_split no son predictores.
id_columns = {
    'score', 'student_id', 'source_split',
    'unit_test_assignment_log_id', 'problem_id'
}
feature_columns = [c for c in df.columns if c not in id_columns]
X = df[feature_columns].apply(pd.to_numeric, errors='coerce').fillna(-1).astype('float32')
y = df['score'].astype('int8')

# Se usan directamente las particiones oficiales source_split.
train_mask = df['source_split'].eq('train')
eval_mask = df['source_split'].eq('evaluation')
X_train = X.loc[train_mask].reset_index(drop=True)
X_test = X.loc[eval_mask].reset_index(drop=True)
y_train = y.loc[train_mask].to_numpy()
y_test = y.loc[eval_mask].to_numpy()

train_students = set(df.loc[train_mask, 'student_id'].dropna())
eval_students = set(df.loc[eval_mask, 'student_id'].dropna())
overlap = train_students & eval_students
shared_eval_rows = df.loc[eval_mask, 'student_id'].isin(train_students).sum()

print(f'Predictores: {len(feature_columns)} | train: {len(y_train):,} | evaluation: {len(y_test):,}')
print(f'Estudiantes compartidos: {len(overlap)} | registros evaluation de esos estudiantes: {shared_eval_rows:,}')


## 3. Modelos e hiperparámetros

Random Forest práctica reproduce la configuración usada en la práctica 1-2: 600 árboles, profundidad máxima 19, mínimo 10 observaciones para dividir, mínimo 2 por hoja, max_features=sqrt, bootstrap y semilla 42.

Random Forest modificado conserva el algoritmo y cambia profundidad a 12, min_samples_split a 20 y min_samples_leaf a 5. Son parámetros manuales para comparar árboles más restringidos; no se presentan como óptimos.

Extra Trees se compara en el artículo original [1]. Usa la misma muestra, predictores, partición, cantidad de árboles y límites del Random Forest base. Extra Trees incorpora aleatoriedad en los umbrales de división, además de la selección aleatoria de predictores.

En los tres modelos, n_jobs=-1 utiliza los núcleos disponibles y random_state=42 fija la aleatoriedad. No se realiza búsqueda automática de parámetros ni ajuste del umbral.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

SEED = 42
N_ESTIMATORS = 600

models = {
    'RF práctica': RandomForestClassifier(
        n_estimators=N_ESTIMATORS, # cantidad de árboles de la práctica
        max_depth=19,              # profundidad máxima de la práctica
        min_samples_split=10,      # mínimo de datos para dividir un nodo
        min_samples_leaf=2,        # mínimo de datos en una hoja
        max_features='sqrt',       # candidatos por división: raíz del número de predictores
        bootstrap=True,            # cada árbol usa una muestra con reemplazo
        random_state=SEED,         # hace reproducible el ajuste
        n_jobs=-1                  # usa todos los núcleos disponibles
    ),
    'RF parámetros modificados': RandomForestClassifier(
        n_estimators=N_ESTIMATORS, # cantidad de árboles controlada
        max_depth=12,              # limita la complejidad de cada árbol
        min_samples_split=20,      # exige más datos antes de dividir
        min_samples_leaf=5,        # evita hojas con muy pocos registros
        max_features='sqrt',       # conserva la selección de predictores
        bootstrap=True,
        random_state=SEED,
        n_jobs=-1
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=19,
        min_samples_split=10,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,            # iguala el muestreo con el RF base
        random_state=SEED,
        n_jobs=-1
    )
}


## 4. Entrenamiento y métricas solicitadas

Cada modelo se ajusta con `X_train` e `y_train`. Se calculan únicamente las cuatro métricas indicadas: accuracy, precisión macro, recall macro y F1 macro. Accuracy es la proporción total de predicciones correctas. Precisión indica, para cada clase, qué proporción de las predicciones asignadas a esa clase fue correcta; recall indica qué proporción de los casos reales de cada clase se identificó; F1 combina precisión y recall.

El asterisco identifica el promedio macro: la métrica se calcula por separado para `score=0` y `score=1`, y luego se promedian ambas clases con el mismo peso. No se presentan columnas separadas por clase ni se calcula el promedio ponderado. El asterisco no indica significancia estadística.

Se presentan dos tablas: entrenamiento, calculada sobre los registros usados para ajustar los modelos, y evaluación (prueba), calculada sobre los registros reservados. La tabla de evaluación es la referencia principal para comparar los tres modelos.


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, confusion_matrix
)

train_rows = []
evaluation_rows = []
roc_data = {}
confusion_data = {}

for model_name, model in models.items():
    # El modelo aprende solamente de la partición de entrenamiento.
    model.fit(X_train, y_train)
    roc_data[model_name] = {}
    confusion_data[model_name] = {}

    for split_name, table_rows, X_part, y_part in [
        ('train', train_rows, X_train, y_train),
        ('evaluation', evaluation_rows, X_test, y_test)
    ]:
        predicted = model.predict(X_part)
        table_rows.append({
            'Algoritmo': model_name,
            'Accuracy': accuracy_score(y_part, predicted),
            'Precisión*': precision_score(
                y_part, predicted, average='macro', zero_division=0
            ),
            'Recall*': recall_score(
                y_part, predicted, average='macro', zero_division=0
            ),
            'F1-score*': f1_score(
                y_part, predicted, average='macro', zero_division=0
            )
        })

        # Los siguientes resultados se usan solo para dibujar las gráficas.
        confusion_data[model_name][split_name] = confusion_matrix(
            y_part, predicted, labels=[0, 1]
        )
        class_index = {int(label): i for i, label in enumerate(model.classes_)}
        probability_score0 = model.predict_proba(X_part)[:, class_index[0]]
        actual_score0 = (y_part == 0).astype('int8')
        fpr, tpr, _ = roc_curve(actual_score0, probability_score0)
        auc_score0 = roc_auc_score(actual_score0, probability_score0)
        roc_data[model_name][split_name] = (fpr, tpr, auc_score0)

train_metrics = pd.DataFrame(train_rows)
evaluation_metrics = pd.DataFrame(evaluation_rows)

display(Markdown(f'### Tabla: métricas de entrenamiento *(n = {len(y_train):,})*'))
display(train_metrics.round(4))
display(Markdown(f'### Tabla: métricas de evaluación o prueba *(n = {len(y_test):,})*'))
display(evaluation_metrics.round(4))
display(Markdown('**Nota:** el asterisco (`*`) indica promedio macro: se promedian `score=0` y `score=1` con el mismo peso.'))

train_metrics.to_csv('metricas_entrenamiento.csv', index=False, float_format='%.4f')
evaluation_metrics.to_csv('metricas_evaluacion.csv', index=False, float_format='%.4f')
print('Guardados: metricas_entrenamiento.csv y metricas_evaluacion.csv')


## 5. Cómo interpretar las tablas

- **Entrenamiento:** muestra cómo se ajusta cada modelo con los registros usados para aprender. Puede tener valores mayores porque esos registros ya fueron vistos por el modelo.
- **Evaluación o prueba:** muestra los resultados en registros reservados. Use esta tabla para comparar los modelos en esta corrida.
- **Promedio macro (*):** promedia las métricas de `score=0` y `score=1` por igual. Así, la clase más numerosa no domina el promedio. La tabla no desglosa las métricas por clase.

La partición oficial separa registros, pero hay 31 estudiantes presentes en ambos conjuntos (552 registros de evaluación). Por ello, los resultados no demuestran el desempeño en estudiantes completamente nuevos; para esa afirmación se necesita una validación agrupada por estudiante.


## 6. Gráfico comparativo de las métricas

La figura reúne entrenamiento y evaluación en dos paneles, con la misma escala de 0 a 1. Cada color representa una de las cuatro métricas; los valores macro promedian `score=0` y `score=1` con igual peso. La evaluación permite comparar los modelos en registros reservados; entrenamiento muestra el ajuste a los datos usados para aprender.


In [ ]:
# Comparación dinámica de las cuatro métricas acordadas.
# Se usan las tablas calculadas arriba; no se vuelven a entrenar los modelos.
metric_specs = [
    ('Accuracy', 'Accuracy', '#2a6f97'),
    ('Precisión*', 'Precisión macro*', '#ee9b00'),
    ('Recall*', 'Recall macro*', '#2a9d8f'),
    ('F1-score*', 'F1 macro*', '#9b5de5')
]
model_order = list(models.keys())
x = np.arange(len(model_order))
bar_width = 0.19

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), dpi=180, sharey=True)
for ax, metrics_table, panel_title in [
    (axes[0], train_metrics, 'Entrenamiento'),
    (axes[1], evaluation_metrics, 'Evaluación (prueba)')
]:
    indexed = metrics_table.set_index('Algoritmo').loc[model_order]
    for metric_index, (column, label, color) in enumerate(metric_specs):
        offset = (metric_index - (len(metric_specs) - 1) / 2) * bar_width
        bars = ax.bar(
            x + offset, indexed[column].to_numpy(), width=bar_width,
            label=label, color=color
        )
        ax.bar_label(bars, fmt='%.2f', fontsize=7, padding=2)
    ax.set_title(panel_title)
    ax.set_xticks(x, model_order, rotation=8)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Valor de la métrica')
    ax.grid(axis='y', alpha=0.25)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Comparación de las cuatro métricas', y=1.10)
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig('comparacion_metricas_train_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Guardado: comparacion_metricas_train_evaluation.png')


## 7. Curvas ROC y matrices de confusión

Las tablas comparan únicamente accuracy, precisión macro, recall macro y F1 macro. Las siguientes figuras ayudan a visualizar los resultados: la curva ROC muestra el cambio entre falsos positivos y verdaderos positivos al variar el umbral; las matrices muestran aciertos y errores en la evaluación. Las únicas métricas tabuladas son las cuatro solicitadas.

En la ROC, `score=0` se considera la clase de interés. La diagonal representa un resultado equivalente al azar. El AUC aparece solo en la leyenda de la ROC, no en las tablas de métricas. En cada matriz, las filas son la clase real y las columnas la predicha: arriba a la izquierda están los `score=0` acertados; arriba a la derecha, los `score=0` clasificados como 1; abajo a la izquierda, los `score=1` clasificados como 0; abajo a la derecha, los `score=1` acertados.


In [ ]:
colors = {
    'RF práctica': '#1f4e79',
    'RF parámetros modificados': '#d28e00',
    'Extra Trees': '#24805a'
}
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=180)
for ax, split_name, title in zip(
    axes, ['train', 'evaluation'], ['Entrenamiento', 'Evaluación (prueba)']
):
    for model_name in models:
        fpr, tpr, auc_score0 = roc_data[model_name][split_name]
        ax.plot(
            fpr, tpr, lw=1.8, color=colors[model_name],
            label=f'{model_name} (AUC={auc_score0:.3f})'
        )
    ax.plot([0, 1], [0, 1], '--', color='gray', lw=1, label='Referencia al azar')
    ax.set(
        xlim=(0, 1), ylim=(0, 1.02),
        xlabel='Tasa de falsos positivos',
        ylabel='Tasa de verdaderos positivos', title=title
    )
    ax.grid(alpha=.2)
    ax.legend(fontsize=7, loc='lower right')
fig.suptitle('Curvas ROC para score=0')
fig.tight_layout()
fig.savefig('curvas_ROC_train_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Guardado: curvas_ROC_train_evaluation.png')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), dpi=180)
for ax, model_name in zip(axes, models):
    matrix = confusion_data[model_name]['evaluation']
    ax.imshow(matrix, cmap='Blues')
    ax.set_title(model_name, fontsize=9)
    ax.set_xticks([0, 1], ['Predicho 0', 'Predicho 1'], fontsize=7)
    ax.set_yticks([0, 1], ['Real 0', 'Real 1'], fontsize=7)
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(
            col, row, f'{value:,}', ha='center', va='center', fontsize=8,
            color='white' if value > matrix.max() * .55 else 'black'
        )
fig.suptitle('Matrices de confusión en evaluación (prueba)')
fig.tight_layout()
fig.savefig('matrices_confusion_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Guardado: matrices_confusion_evaluation.png')


## Referencia

[1] A. Yaramala, S. Farokhi y H. Karimi, “Navigating the Data-Rich Landscape of Online Learning: Insights and Predictions from ASSISTments,” Proceedings of the 17th International Conference on Educational Data Mining, pp. 321–331, Atlanta, Georgia, EE. UU., jul. 2024.
